# Diagnose a vLLM setup on a free GPU

What happens in this notebook:

- A deliberately throttled vLLM setup is measured.
- Tensward names the bottleneck and suggests a change.
- The change is measured, and Tensward shows what changed.

**Before you start:** Runtime → Change runtime type → T4 GPU. Then Runtime → Run all. It takes about 15 minutes.

In [ ]:
import shutil, subprocess, sys
if shutil.which("nvidia-smi") is None or subprocess.run(["nvidia-smi"], capture_output=True).returncode != 0:
    sys.exit("No GPU in this runtime. Choose Runtime → Change runtime type → T4 GPU, then run all again.")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout)

## Install

Installing vLLM replaces Colab's PyTorch and takes a few minutes. pip may print dependency-conflict warnings about other preinstalled Colab packages; they do not affect this notebook. Colab's `torchaudio` is built for the old PyTorch, so the cell removes it (vLLM does not need it here). `tensward env` will say Docker is not available: that is expected on Colab, where Tensward runs vLLM directly (`--runtime local`).

The `run` helper runs a shell command, shows its output as it goes, and stops the notebook if the command fails.

In [ ]:
import pathlib, re, subprocess, sys
def run(command: str) -> None:
    """Run a shell command, showing its output as it goes; stop the notebook if it fails."""
    print("$", command, flush=True)
    process = subprocess.Popen(
        command, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    for line in process.stdout:
        print(line, end="", flush=True)
    code = process.wait()
    if code != 0:
        sys.exit(f"That command failed (exit {code}); its error is above.")
run('pip -q install --progress-bar off "tensward>=0.3.2" "vllm==0.30.0"')
run("pip -q uninstall -y torchaudio")
run("tensward env")

## Download a model

Qwen2.5-1.5B-Instruct is small enough for the T4's 16 GB.

In [ ]:
from huggingface_hub import snapshot_download
from huggingface_hub.utils import disable_progress_bars, logging

disable_progress_bars()
logging.set_verbosity_error()
print("Downloading Qwen2.5-1.5B-Instruct (about 3 GB)...", flush=True)
snapshot_download("Qwen/Qwen2.5-1.5B-Instruct", local_dir="/content/model")
print("Model downloaded to /content/model")

## Workload

A workload is a serving configuration plus a file of prompts. This one sends 96 requests with 32 clients at a time, and adapts the example configuration to this model and to the throttled setup below.

In [ ]:
import json, pathlib, urllib.request
TAG = "v0.3.2"
work = pathlib.Path("/content/tensward"); work.mkdir(exist_ok=True)
urllib.request.urlretrieve(
    f"https://raw.githubusercontent.com/Tensward/tensward/{TAG}/examples/prompts.jsonl",
    work / "prompts.jsonl",
)
config = json.loads(urllib.request.urlopen(
    f"https://raw.githubusercontent.com/Tensward/tensward/{TAG}/examples/config.json").read())
config["case"].update({"weight_precision": "bf16", "activation_dtype": "bfloat16", "max_model_len": 4096,
                       "max_num_seqs": 2, "gpu_memory_utilization": 0.85})
config["workload"].update({"request_count": 96, "arrival": {"kind": "closed_loop", "concurrency": 32}})
(work / "config.json").write_text(json.dumps(config, indent=1))

## Register your current setup

`--max-num-seqs 2` stands in for your own setup: it lets only 2 requests run at once. The T4 has no bf16 support, so the command serves in float16 with `--dtype float16`.

In [ ]:
import time
project = f"/content/tensward/project-{int(time.time())}"
current = ("vllm serve /content/model --max-num-seqs 2 --dtype float16 --max-model-len 4096 "
           "--enable-auto-tool-choice --tool-call-parser hermes")
run(f"tensward init --project {project} --model /content/model --config /content/tensward/config.json "
    f"--prompts /content/tensward/prompts.jsonl --current '{current}' > /content/tensward/init.json")
print("Registered. The full record is in /content/tensward/init.json.")

## Analyse

Tensward starts your current setup, sends the workload, and reads the engine's metrics.

In [ ]:
run(f"tensward analyse --project {project} --runtime local")
def newest_report(project):
    runs = sorted(pathlib.Path(project, "runs").iterdir(), key=lambda p: p.name)
    return (runs[-1] / "report.md").read_text()
def section(report, *titles):
    parts = re.split(r"(?m)^## ", report)
    return "\n".join("## " + p for p in parts if p.split("\n", 1)[0].startswith(titles))
report = newest_report(project)
print(section(report, "Diagnosis"))

## Try the first suggestion

The next cell runs the first `Try:` command from the report and prints what changed.

Answers at temperature 0 can change when more requests share a batch, because vLLM is not batch-invariant. The comparison may therefore say "Answers changed"; look at the pairs it shows.

The report's batch-invariant advice needs a newer GPU than the T4.

In [ ]:
match = re.search(r"Try: `(tensward analyse [^`]+)`", report)
if match is None:
    print("Tensward suggested no change for this setup.")
else:
    run(match.group(1) + " --runtime local")
    print(section(newest_report(project), "Diagnosis"))

## Next steps

Register your own model and setup: pass `--current` with the exact command you serve with. See the README sections on [installing](https://github.com/Tensward/tensward#install), [`init`](https://github.com/Tensward/tensward#2-register-the-project) and [`analyse`](https://github.com/Tensward/tensward#3-analyse).

Found something, right or wrong? Open an issue: https://github.com/Tensward/tensward/issues